# MLOPS_EL — Free CUDA benchmark

This notebook benchmarks **YOLO11m at 960 pixels, batch 4, for two engineering-only epochs** on 256 training images and 32 other training images. No test data. It does **not** authorize full training or claim 70% mAP.

Use a **free GPU runtime** (Runtime → Change runtime type → T4 GPU). If no free GPU is available, stop; do not buy compute. Upload only the benchmark archive when asked. Google Drive is used for this project’s checkpoint recovery, not as the image training disk.

Dataset: the versioned `plitterstreet-labels-v3` derivative; source attribution: https://zenodo.org/records/8288500 (CC BY 4.0).

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a free GPU runtime before continuing; no CPU fallback.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


In [ ]:
import sys, subprocess, pathlib, importlib.metadata
constraints=pathlib.Path('/content/cuda-preserve.txt')
constraints.write_text('torch=='+importlib.metadata.version('torch')+'\n'+'torchvision=='+importlib.metadata.version('torchvision')+'\n')
requirements=pathlib.Path('/content/requirements-cloud.txt')
requirements.write_text("# Keep the runtime's working CUDA torch/torchvision builds; record their exact versions.\nultralytics==8.4.201\nmlflow==3.17.2\nPyYAML==6.0.3\n")
requirements.write_text(requirements.read_text() + 'jedi==0.19.2\n')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade-strategy', 'only-if-needed', '-c', str(constraints), '-r', str(requirements)], check=True)
check = subprocess.run([sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True)
print(check.stdout, check.stderr)
check.check_returncode()
print('Dependencies checked; the runtime CUDA Torch build was preserved.')


In [ ]:
import base64, pathlib, sys, os
os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
os.environ['YOLO_CONFIG_DIR']='/content/mlops_yolo_settings'
os.environ['MPLCONFIGDIR']='/content/mlops_matplotlib'
pathlib.Path('/content/mlops_cloud_runner.py').write_bytes(base64.b64decode('IiIiRnJlZS1DVURBIGJlbmNobWFyay90cmFpbmluZyBoZWxwZXJzLiBGdWxsIHRyYWluaW5nIGlzIGV4cGxpY2l0bHkgZ2F0ZWQuIiIiCmltcG9ydCBhcmdwYXJzZQpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBDb3VudGVyCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgsIFB1cmVQb3NpeFBhdGgKaW1wb3J0IHBsYXRmb3JtCmltcG9ydCBzaHV0aWwKaW1wb3J0IHNxbGl0ZTMKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHRhcmZpbGUKaW1wb3J0IHRpbWUKCgpkZWYgc2hhKHBhdGgpOgogICAgaD1oYXNobGliLnNoYTI1NigpCiAgICB3aXRoIFBhdGgocGF0aCkub3BlbigncmInKSBhcyBmOgogICAgICAgIGZvciBibG9jayBpbiBpdGVyKGxhbWJkYTpmLnJlYWQoMTAyNCoxMDI0KSxiJycpOmgudXBkYXRlKGJsb2NrKQogICAgcmV0dXJuIGguaGV4ZGlnZXN0KCkKCgpkZWYgc2FmZV9uYW1lKG5hbWUpOgogICAgcD1QdXJlUG9zaXhQYXRoKG5hbWUpCiAgICBpZiBwLmlzX2Fic29sdXRlKCkgb3IgJy4uJyBpbiBwLnBhcnRzIG9yIG5vdCBwLnBhcnRzIG9yICdcXCcgaW4gbmFtZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdVbnNhZmUgYXJjaGl2ZSBwYXRoJykKICAgIHJldHVybiBwCgoKZGVmIHVucGFjayhhcmNoaXZlLGRlc3RpbmF0aW9uLGV4cGVjdGVkX3NoYSk6CiAgICBhcmNoaXZlLGRlc3RpbmF0aW9uPVBhdGgoYXJjaGl2ZSksUGF0aChkZXN0aW5hdGlvbikKICAgIGlmIHNoYShhcmNoaXZlKSE9ZXhwZWN0ZWRfc2hhOnJhaXNlIFZhbHVlRXJyb3IoJ0FyY2hpdmUgY2hlY2tzdW0gbWlzbWF0Y2gnKQogICAgaWYgZGVzdGluYXRpb24uZXhpc3RzKCk6cmFpc2UgVmFsdWVFcnJvcignRXh0cmFjdGlvbiBkZXN0aW5hdGlvbiBleGlzdHM7IHVzZSBhIGZyZXNoIGRpcmVjdG9yeScpCiAgICB3aXRoIHRhcmZpbGUub3BlbihhcmNoaXZlLCdyOicpIGFzIHRhcjoKICAgICAgICBtZW1iZXJzPXRhci5nZXRtZW1iZXJzKCk7bmFtZXM9W20ubmFtZSBmb3IgbSBpbiBtZW1iZXJzXQogICAgICAgIGlmIGxlbihuYW1lcykhPWxlbihzZXQobmFtZXMpKTpyYWlzZSBWYWx1ZUVycm9yKCdEdXBsaWNhdGUgYXJjaGl2ZSBlbnRyaWVzJykKICAgICAgICBmb3IgbSBpbiBtZW1iZXJzOgogICAgICAgICAgICBzYWZlX25hbWUobS5uYW1lKQogICAgICAgICAgICBpZiBub3QgbS5pc2ZpbGUoKTpyYWlzZSBWYWx1ZUVycm9yKCdPbmx5IHJlZ3VsYXIgZmlsZXMgYXJlIGFjY2VwdGVkOyBsaW5rcyBhcmUgcmVqZWN0ZWQnKQogICAgICAgIG1hbmlmZXN0PWpzb24ubG9hZCh0YXIuZXh0cmFjdGZpbGUoJ21hbmlmZXN0Lmpzb24nKSkKICAgICAgICBleHBlY3RlZD17clsncGF0aCddOnIgZm9yIHIgaW4gbWFuaWZlc3RbJ2ZpbGVzJ119CiAgICAgICAgaWYgbGVuKGV4cGVjdGVkKSE9bGVuKG1hbmlmZXN0WydmaWxlcyddKTpyYWlzZSBWYWx1ZUVycm9yKCdEdXBsaWNhdGUgbWFuaWZlc3QgcGF0aHMnKQogICAgICAgIGlmIHNldChuYW1lcykhPXNldChleHBlY3RlZCl8eydtYW5pZmVzdC5qc29uJ306cmFpc2UgVmFsdWVFcnJvcignQXJjaGl2ZSBtZW1iZXJzIGRpZmZlciBmcm9tIG1hbmlmZXN0JykKICAgICAgICBkZXN0aW5hdGlvbi5ta2RpcihwYXJlbnRzPVRydWUpCiAgICAgICAgZm9yIG0gaW4gbWVtYmVyczoKICAgICAgICAgICAgaWYgbS5uYW1lPT0nbWFuaWZlc3QuanNvbic6Y29udGludWUKICAgICAgICAgICAgcm93PWV4cGVjdGVkW20ubmFtZV0KICAgICAgICAgICAgaWYgcm93WydieXRlcyddIT1tLnNpemU6cmFpc2UgVmFsdWVFcnJvcignQXJjaGl2ZSBzaXplIG1pc21hdGNoJykKICAgICAgICAgICAgdGFyZ2V0PWRlc3RpbmF0aW9uLmpvaW5wYXRoKCpzYWZlX25hbWUobS5uYW1lKS5wYXJ0cyk7dGFyZ2V0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICAgICAgaD1oYXNobGliLnNoYTI1NigpCiAgICAgICAgICAgIHdpdGggdGFyLmV4dHJhY3RmaWxlKG0pIGFzIHNyYyx0YXJnZXQub3Blbignd2InKSBhcyBkc3Q6CiAgICAgICAgICAgICAgICBmb3IgYmxvY2sgaW4gaXRlcihsYW1iZGE6c3JjLnJlYWQoMTAyNCoxMDI0KSxiJycpOgogICAgICAgICAgICAgICAgICAgIGgudXBkYXRlKGJsb2NrKTtkc3Qud3JpdGUoYmxvY2spCiAgICAgICAgICAgIGlmIGguaGV4ZGlnZXN0KCkhPXJvd1snc2hhMjU2J106cmFpc2UgVmFsdWVFcnJvcihmJ0ZpbGUgY2hlY2tzdW0gbWlzbWF0Y2g6IHttLm5hbWV9JykKICAgICAgICAoZGVzdGluYXRpb24vJ21hbmlmZXN0Lmpzb24nKS53cml0ZV90ZXh0KGpzb24uZHVtcHMobWFuaWZlc3QsaW5kZW50PTIpKQogICAgdmVyaWZ5X2J1bmRsZShkZXN0aW5hdGlvbikKICAgIHJldHVybiBtYW5pZmVzdAoKCmRlZiB2ZXJpZnlfYnVuZGxlKHJvb3QpOgogICAgcm9vdD1QYXRoKHJvb3QpO21hbmlmZXN0PWpzb24ubG9hZHMoKHJvb3QvJ21hbmlmZXN0Lmpzb24nKS5yZWFkX3RleHQoKSkKICAgIGZvciByb3cgaW4gbWFuaWZlc3RbJ2ZpbGVzJ106CiAgICAgICAgcGF0aD1yb290LmpvaW5wYXRoKCpzYWZlX25hbWUocm93WydwYXRoJ10pLnBhcnRzKQogICAgICAgIGlmIG5vdCBwYXRoLmlzX2ZpbGUoKSBvciBwYXRoLnN0YXQoKS5zdF9zaXplIT1yb3dbJ2J5dGVzJ10gb3Igc2hhKHBhdGgpIT1yb3dbJ3NoYTI1NiddOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYnQnVuZGxlIHZlcmlmaWNhdGlvbiBmYWlsZWQ6IHtyb3dbInBhdGgiXX0nKQogICAgaWYgYW55KCcvdGVzdC8nIGluIHJbJ3BhdGgnXSBmb3IgciBpbiBtYW5pZmVzdFsnZmlsZXMnXSk6cmFpc2UgVmFsdWVFcnJvcignVGVzdCBkYXRhIG11c3Qgbm90IGJlIGV4cG9ydGVkJykKICAgIGV4cGVjdGVkPXsnYmVuY2htYXJrJzooMjU2LDMyKSwndHJhaW52YWwnOig3MDAwLDE1MDApfVttYW5pZmVzdFsnc2NvcGUnXV0KICAgIGlmIChtYW5pZmVzdFsndHJhaW5faW1hZ2VzJ10sbWFuaWZlc3RbJ3ZhbF9pbWFnZXMnXSkhPWV4cGVjdGVkOnJhaXNlIFZhbHVlRXJyb3IoJ1VuZXhwZWN0ZWQgZGF0YXNldCBzaXplJykKICAgIGNvdW50cz1Db3VudGVyKCcvJy5qb2luKHJbJ3BhdGgnXS5zcGxpdCgnLycpWzozXSkgZm9yIHIgaW4gbWFuaWZlc3RbJ2ZpbGVzJ10pCiAgICBmb3Igc3BsaXQsY291bnQgaW4gemlwKCgndHJhaW4nLCd2YWwnKSxleHBlY3RlZCk6CiAgICAgICAgaWYgY291bnRzW2YnZGF0YXNldC9pbWFnZXMve3NwbGl0fSddIT1jb3VudCBvciBjb3VudHNbZidkYXRhc2V0L2xhYmVscy97c3BsaXR9J10hPWNvdW50OgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdBY3R1YWwgaW1hZ2UvbGFiZWwgY291bnRzIGRpZmZlciBmcm9tIGRlY2xhcmVkIHNjb3BlJykKICAgIHJldHVybiBtYW5pZmVzdAoKCmRlZiBhdG9taWNfY29weShzb3VyY2UsZGVzdGluYXRpb24pOgogICAgZGVzdGluYXRpb249UGF0aChkZXN0aW5hdGlvbik7ZGVzdGluYXRpb24ucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogICAgdGVtcG9yYXJ5PWRlc3RpbmF0aW9uLndpdGhfbmFtZShkZXN0aW5hdGlvbi5uYW1lKycucGFydCcpCiAgICBzaHV0aWwuY29weWZpbGUoc291cmNlLHRlbXBvcmFyeSkKICAgIGlmIHNoYSh0ZW1wb3JhcnkpIT1zaGEoc291cmNlKTpyYWlzZSBJT0Vycm9yKCdQZXJzaXN0ZW50IGNvcHkgdmVyaWZpY2F0aW9uIGZhaWxlZCcpCiAgICB0ZW1wb3JhcnkucmVwbGFjZShkZXN0aW5hdGlvbikKCgpkZWYgc2F2ZV9jaGVja3BvaW50X2dlbmVyYXRpb24odHJhaW5lcixwZXJzaXN0ZW50LG1ldGFkYXRhKToKICAgICIiIktlZXAgdHdvIGNvbXBsZXRlIGNoZWNrcG9pbnQgZ2VuZXJhdGlvbnM7IHB1Ymxpc2ggcG9pbnRlciBvbmx5IGFmdGVyIHZlcmlmaWVkIGNvcGllcy4iIiIKICAgIHBlcnNpc3RlbnQ9UGF0aChwZXJzaXN0ZW50KTtiYXNlPXBlcnNpc3RlbnQvJ2NoZWNrcG9pbnRzJztiYXNlLm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogICAgbmFtZT1mJ2Vwb2NoLXtpbnQodHJhaW5lci5lcG9jaCkrMTowNGR9JwogICAgZ2VuZXJhdGlvbj1iYXNlL25hbWU7Z2VuZXJhdGlvbi5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgZmlsZXM9e30KICAgIGZvciBrZXkgaW4gKCdsYXN0JywnYmVzdCcpOgogICAgICAgIHNvdXJjZT1QYXRoKGdldGF0dHIodHJhaW5lcixrZXkpKQogICAgICAgIGlmIHNvdXJjZS5leGlzdHMoKToKICAgICAgICAgICAgdGFyZ2V0PWdlbmVyYXRpb24vZid7a2V5fS5wdCc7YXRvbWljX2NvcHkoc291cmNlLHRhcmdldCk7ZmlsZXNba2V5XT1zaGEodGFyZ2V0KQogICAgaWYgJ2xhc3QnIG5vdCBpbiBmaWxlczpyYWlzZSBJT0Vycm9yKCdObyByZXN1bWFibGUgY2hlY2twb2ludCB3YXMgc2F2ZWQnKQogICAgcmVjb3JkPXsqKm1ldGFkYXRhLCdlcG9jaCc6aW50KHRyYWluZXIuZXBvY2gpKzEsJ2ZpbGVzJzpmaWxlcywnZ2VuZXJhdGlvbic6bmFtZX0KICAgIHRlbXBvcmFyeT1nZW5lcmF0aW9uLydzdGF0ZS5qc29uLnBhcnQnO3RlbXBvcmFyeS53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVjb3JkLGluZGVudD0yKSk7dGVtcG9yYXJ5LnJlcGxhY2UoZ2VuZXJhdGlvbi8nc3RhdGUuanNvbicpCiAgICBwb2ludGVyPWJhc2UvJ2xhdGVzdC5qc29uLnBhcnQnO3BvaW50ZXIud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlY29yZCxpbmRlbnQ9MikpO3BvaW50ZXIucmVwbGFjZShiYXNlLydsYXRlc3QuanNvbicpCiAgICAjIE9ubHkgZ2VuZXJhdGVkIHByb2plY3QgY2hlY2twb2ludHMgdW5kZXIgdGhpcyBtb2RlIGRpcmVjdG9yeSBhcmUgZWxpZ2libGUgZm9yIHJvdGF0aW9uLgogICAgY29tcGxldGU9c29ydGVkKFtwIGZvciBwIGluIGJhc2UuZ2xvYignZXBvY2gtKicpIGlmIChwLydzdGF0ZS5qc29uJykuaXNfZmlsZSgpXSkKICAgIGZvciBvbGQgaW4gY29tcGxldGVbOi0yXTpzaHV0aWwucm10cmVlKG9sZCkKICAgIHJldHVybiByZWNvcmQKCgpkZWYgdmVyaWZ5X2xhdGVzdChwZXJzaXN0ZW50KToKICAgIGJhc2U9UGF0aChwZXJzaXN0ZW50KS8nY2hlY2twb2ludHMnCiAgICBzdGF0ZT1qc29uLmxvYWRzKChiYXNlLydsYXRlc3QuanNvbicpLnJlYWRfdGV4dCgpKQogICAgZ2VuZXJhdGlvbj1iYXNlL3N0YXRlWydnZW5lcmF0aW9uJ10KICAgIGZvciBrZXksZXhwZWN0ZWQgaW4gc3RhdGVbJ2ZpbGVzJ10uaXRlbXMoKToKICAgICAgICBpZiBzaGEoZ2VuZXJhdGlvbi9mJ3trZXl9LnB0JykhPWV4cGVjdGVkOnJhaXNlIFZhbHVlRXJyb3IoJ1NhdmVkIGNoZWNrcG9pbnQgZmFpbGVkIHJlY292ZXJ5IGNoZWNrc3VtJykKICAgIHJldHVybiBnZW5lcmF0aW9uLydsYXN0LnB0JyxzdGF0ZQoKCmRlZiBzbmFwc2hvdF90cmFja2luZyhkYXRhYmFzZSxhcnRpZmFjdF9kaXIscGVyc2lzdGVudCk6CiAgICBwZXJzaXN0ZW50PVBhdGgocGVyc2lzdGVudCk7cGVyc2lzdGVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgIGlmIFBhdGgoZGF0YWJhc2UpLmV4aXN0cygpOgogICAgICAgIHRlbXBvcmFyeT1wZXJzaXN0ZW50LydtbGZsb3cuZGIucGFydCc7dGVtcG9yYXJ5LnVubGluayhtaXNzaW5nX29rPVRydWUpCiAgICAgICAgd2l0aCBzcWxpdGUzLmNvbm5lY3QoZGF0YWJhc2UpIGFzIHNyYyxzcWxpdGUzLmNvbm5lY3QodGVtcG9yYXJ5KSBhcyBkc3Q6c3JjLmJhY2t1cChkc3QpCiAgICAgICAgdGVtcG9yYXJ5LnJlcGxhY2UocGVyc2lzdGVudC8nbWxmbG93LmRiJykKICAgIGlmIFBhdGgoYXJ0aWZhY3RfZGlyKS5leGlzdHMoKToKICAgICAgICBzaHV0aWwuY29weXRyZWUoYXJ0aWZhY3RfZGlyLHBlcnNpc3RlbnQvJ21sYXJ0aWZhY3RzJyxkaXJzX2V4aXN0X29rPVRydWUpCgoKZGVmIHJ1bihyb290LHBlcnNpc3RlbnQsbW9kZT0nYmVuY2htYXJrJyxhcHByb3ZlX2Z1bGw9RmFsc2UscmVzdW1lPUZhbHNlKToKICAgIHJvb3QscGVyc2lzdGVudD1QYXRoKHJvb3QpLnJlc29sdmUoKSxQYXRoKHBlcnNpc3RlbnQpLnJlc29sdmUoKQogICAgbWFuaWZlc3Q9dmVyaWZ5X2J1bmRsZShyb290KQogICAgZXhwZWN0ZWRfc2NvcGU9J2JlbmNobWFyaycgaWYgbW9kZT09J2JlbmNobWFyaycgZWxzZSAndHJhaW52YWwnCiAgICBpZiBtYW5pZmVzdFsnc2NvcGUnXSE9ZXhwZWN0ZWRfc2NvcGU6cmFpc2UgVmFsdWVFcnJvcignV3JvbmcgYnVuZGxlIHNjb3BlIGZvciByZXF1ZXN0ZWQgcnVuJykKICAgIGlmIG1vZGU9PSd0cmFpbicgYW5kIG5vdCBhcHByb3ZlX2Z1bGw6cmFpc2UgVmFsdWVFcnJvcignRnVsbCB0cmFpbmluZyBhcHByb3ZhbCBpcyByZXF1aXJlZDsgYmVuY2htYXJrIG9ubHkgaXMgY3VycmVudGx5IGF1dGhvcml6ZWQnKQogICAgIyBUaGUgbW91bnRlZCBwYXRoIG11c3QgYmUgb3V0c2lkZSBlcGhlbWVyYWwgL2NvbnRlbnQuIEEgbW91bnQtbGV2ZWwgY2hlY2sgZm9sbG93cyBiZWxvdy4KICAgIGlmIG5vdCBzdHIocGVyc2lzdGVudCkuc3RhcnRzd2l0aCgnL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS8nKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdVc2UgYSBwcm9qZWN0IGRpcmVjdG9yeSB1bmRlciBtb3VudGVkIEdvb2dsZSBEcml2ZSBNeURyaXZlJykKICAgIGlmIG5vdCBvcy5wYXRoLmlzbW91bnQoJy9jb250ZW50L2RyaXZlJyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignR29vZ2xlIERyaXZlIGlzIG5vdCBtb3VudGVkOyBwZXJzaXN0ZW50IHJlY292ZXJ5IGNhbm5vdCBiZSBwcm9taXNlZCcpCiAgICBpbXBvcnQgeWFtbAogICAgaW1wb3J0IHRvcmNoCiAgICBpbXBvcnQgbWxmbG93CiAgICBmcm9tIHVsdHJhbHl0aWNzIGltcG9ydCBZT0xPLHNldHRpbmdzCiAgICBpZiBub3QgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKTpyYWlzZSBSdW50aW1lRXJyb3IoJ05vIENVREEgR1BVIGFsbG9jYXRlZC4gU3RvcDsgbmV2ZXIgZmFsbCBiYWNrIHRvIE1hYy9DUFUgb3IgYSBwYWlkIHJ1bnRpbWUgYXV0b21hdGljYWxseS4nKQogICAgcGVyc2lzdGVudD1wZXJzaXN0ZW50L21vZGU7cGVyc2lzdGVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgIHdyaXRlX3Byb2JlPXBlcnNpc3RlbnQvJ3BlcnNpc3RlbmNlLXByb2JlLnR4dCc7d3JpdGVfcHJvYmUud3JpdGVfdGV4dCgnbWxvcHMtY2hlY2twb2ludC1wcm9iZScpCiAgICBpZiB3cml0ZV9wcm9iZS5yZWFkX3RleHQoKSE9J21sb3BzLWNoZWNrcG9pbnQtcHJvYmUnOnJhaXNlIElPRXJyb3IoJ1BlcnNpc3RlbnQgc3RvcmFnZSBjaGVjayBmYWlsZWQnKQogICAgd29yaz1QYXRoKCcvY29udGVudC9tbG9wc193b3JrJykvbW9kZTt3b3JrLm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogICAgZGF0YWJhc2U9d29yay8nbWxmbG93LmRiJzthcnRpZmFjdF9kaXI9d29yay8nbWxhcnRpZmFjdHMnCiAgICBpZiByZXN1bWUgYW5kIG5vdCBkYXRhYmFzZS5leGlzdHMoKSBhbmQgKHBlcnNpc3RlbnQvJ21sZmxvdy5kYicpLmV4aXN0cygpOgogICAgICAgIHNodXRpbC5jb3B5MihwZXJzaXN0ZW50LydtbGZsb3cuZGInLGRhdGFiYXNlKQogICAgICAgIGlmIChwZXJzaXN0ZW50LydtbGFydGlmYWN0cycpLmV4aXN0cygpOnNodXRpbC5jb3B5dHJlZShwZXJzaXN0ZW50LydtbGFydGlmYWN0cycsYXJ0aWZhY3RfZGlyLGRpcnNfZXhpc3Rfb2s9VHJ1ZSkKICAgIG1sZmxvdy5zZXRfdHJhY2tpbmdfdXJpKGYnc3FsaXRlOi8vL3tkYXRhYmFzZX0nKQogICAgZXhwZXJpbWVudD1tbGZsb3cuZ2V0X2V4cGVyaW1lbnRfYnlfbmFtZSgncGxhc3RpYy13YXN0ZS1waGFzZTEtY2xvdWQnKQogICAgaWYgZXhwZXJpbWVudCBpcyBOb25lOm1sZmxvdy5jcmVhdGVfZXhwZXJpbWVudCgncGxhc3RpYy13YXN0ZS1waGFzZTEtY2xvdWQnLGFydGlmYWN0X2xvY2F0aW9uPWFydGlmYWN0X2Rpci5hc191cmkoKSkKICAgIG1sZmxvdy5zZXRfZXhwZXJpbWVudCgncGxhc3RpYy13YXN0ZS1waGFzZTEtY2xvdWQnKQogICAgY29uZmlnPXlhbWwuc2FmZV9sb2FkKChyb290Lydjb25maWcueWFtbCcpLnJlYWRfdGV4dCgpKTtwYXJhbXM9ZGljdChjb25maWdbJ3RyYWluJ10pCiAgICBwYXJhbXNbJ21vZGVsJ109c3RyKHJvb3QvJ21vZGVscy95b2xvMTFtLnB0Jyk7cGFyYW1zWydkZXZpY2UnXT0nMCcKICAgIGlmIG1vZGU9PSdiZW5jaG1hcmsnOnBhcmFtcy51cGRhdGUoZXBvY2hzPTIsY2xvc2VfbW9zYWljPTAscGxvdHM9RmFsc2UpCiAgICBkYXRhPXlhbWwuc2FmZV9sb2FkKChyb290LydkYXRhc2V0L2RhdGFzZXQueWFtbCcpLnJlYWRfdGV4dCgpKTtkYXRhWydwYXRoJ109c3RyKHJvb3QvJ2RhdGFzZXQnKQogICAgcnVudGltZT13b3JrLydkYXRhc2V0LnlhbWwnO3J1bnRpbWUud3JpdGVfdGV4dCh5YW1sLnNhZmVfZHVtcChkYXRhKSkKICAgIHNldHRpbmdzLnVwZGF0ZSh7a2V5OkZhbHNlIGZvciBrZXkgaW4gKCdtbGZsb3cnLCd3YW5kYicsJ2NsZWFybWwnLCdjb21ldCcsJ25lcHR1bmUnLCd0ZW5zb3Jib2FyZCcpIGlmIGtleSBpbiBzZXR0aW5nc30pCiAgICBvcy5lbnZpcm9uWydNTEZMT1dfRElTQUJMRV9BR0VOVF9ISU5UJ109JzEnCiAgICBmcmVlemU9c3VicHJvY2Vzcy5jaGVja19vdXRwdXQoW29zLnN5cy5leGVjdXRhYmxlLCctbScsJ3BpcCcsJ2ZyZWV6ZSddLHRleHQ9VHJ1ZSkKICAgICh3b3JrLydyZXF1aXJlbWVudHMtY2xvdWQtcmVzb2x2ZWQudHh0Jykud3JpdGVfdGV4dChmcmVlemUpCiAgICBncHU9eyduYW1lJzp0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKSwndnJhbV9HaUInOnRvcmNoLmN1ZGEuZ2V0X2RldmljZV9wcm9wZXJ0aWVzKDApLnRvdGFsX21lbW9yeS8xMDI0KiozLAogICAgICAgICd0b3JjaCc6dG9yY2guX192ZXJzaW9uX18sJ2N1ZGEnOnRvcmNoLnZlcnNpb24uY3VkYSwncHl0aG9uJzpwbGF0Zm9ybS5weXRob25fdmVyc2lvbigpfQogICAgKHdvcmsvJ2dwdS5qc29uJykud3JpdGVfdGV4dChqc29uLmR1bXBzKGdwdSxpbmRlbnQ9MikpCiAgICBjaGVja3BvaW50PU5vbmU7cnVuX2lkPU5vbmUKICAgIGZpbmdlcnByaW50PXNoYShyb290LydtYW5pZmVzdC5qc29uJykKICAgIGlmIHJlc3VtZToKICAgICAgICBjaGVja3BvaW50LHN0YXRlPXZlcmlmeV9sYXRlc3QocGVyc2lzdGVudCkKICAgICAgICBpZiBzdGF0ZVsnbW9kZSddIT1tb2RlIG9yIHN0YXRlWydidW5kbGVfZmluZ2VycHJpbnQnXSE9ZmluZ2VycHJpbnQ6cmFpc2UgVmFsdWVFcnJvcignUmVjb3ZlcnkgYnVuZGxlL21vZGUgbWlzbWF0Y2gnKQogICAgICAgIHJ1bl9pZD1zdGF0ZVsncnVuX2lkJ10KICAgICAgICByZXN0b3JlZD13b3JrLydydW5zJy9ydW5faWQ7cmVzdG9yZWQubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICAgICAgaWYgKHBlcnNpc3RlbnQvJ3RyYWluaW5nJykuZXhpc3RzKCk6c2h1dGlsLmNvcHl0cmVlKHBlcnNpc3RlbnQvJ3RyYWluaW5nJyxyZXN0b3JlZCxkaXJzX2V4aXN0X29rPVRydWUpCiAgICAgICAgZm9yIGtleSBpbiBzdGF0ZVsnZmlsZXMnXToKICAgICAgICAgICAgYXRvbWljX2NvcHkoY2hlY2twb2ludC5wYXJlbnQvZid7a2V5fS5wdCcscmVzdG9yZWQvJ3dlaWdodHMnL2Yne2tleX0ucHQnKQogICAgd2l0aCBtbGZsb3cuc3RhcnRfcnVuKHJ1bl9pZD1ydW5faWQscnVuX25hbWU9Tm9uZSBpZiByZXN1bWUgZWxzZSBmJ3lvbG8xMW0te21vZGV9LXt0aW1lLnN0cmZ0aW1lKCIlWSVtJWQtJUglTSVTIil9JykgYXMgZXhwZXJpbWVudF9ydW46CiAgICAgICAgcnVuX2lkPWV4cGVyaW1lbnRfcnVuLmluZm8ucnVuX2lkCiAgICAgICAgbWxmbG93LnNldF90YWdzKHsncGhhc2UnOicxJywncHJvZmlsZSc6ZidjbG91ZC17bW9kZX0nLCdkYXRhc2V0X2FwcHJvdmFsJzoncGVuZGluZycsJ2J1bmRsZV9maW5nZXJwcmludCc6ZmluZ2VycHJpbnQsCiAgICAgICAgICAgICdzb3VyY2VfZ2l0X2NvbW1pdCc6bWFuaWZlc3RbJ3NvdXJjZV9naXRfY29tbWl0J10sJ25vdF9mb3JfcXVhbGl0eV9jbGFpbXMnOnN0cihtb2RlPT0nYmVuY2htYXJrJyksJ2dwdSc6Z3B1WyduYW1lJ119KQogICAgICAgIGlmIG5vdCByZXN1bWU6bWxmbG93LmxvZ19wYXJhbXMocGFyYW1zKQogICAgICAgIGZvciBwYXRoIGluIFtyb290LydtYW5pZmVzdC5qc29uJyxyb290Lydjb25maWcueWFtbCcsd29yay8ncmVxdWlyZW1lbnRzLWNsb3VkLXJlc29sdmVkLnR4dCcsd29yay8nZ3B1Lmpzb24nXToKICAgICAgICAgICAgbWxmbG93LmxvZ19hcnRpZmFjdChzdHIocGF0aCksYXJ0aWZhY3RfcGF0aD0ncHJvdmVuYW5jZScpCiAgICAgICAgdG9yY2guY3VkYS5yZXNldF9wZWFrX21lbW9yeV9zdGF0cygpCiAgICAgICAgbW9kZWw9WU9MTyhzdHIoY2hlY2twb2ludCkgaWYgY2hlY2twb2ludCBlbHNlIHBhcmFtc1snbW9kZWwnXSkKICAgICAgICBtZXRhZGF0YT17J21vZGUnOm1vZGUsJ3J1bl9pZCc6cnVuX2lkLCdidW5kbGVfZmluZ2VycHJpbnQnOmZpbmdlcnByaW50fQogICAgICAgIHNhdmVkPVtdO3RpbWVzPVtdO3N0YXJ0PVtOb25lXQogICAgICAgIGRlZiBiYXRjaF9zdGFydCh0cmFpbmVyKToKICAgICAgICAgICAgdG9yY2guY3VkYS5zeW5jaHJvbml6ZSgpO3N0YXJ0WzBdPXRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICBkZWYgYmF0Y2hfZW5kKHRyYWluZXIpOgogICAgICAgICAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCk7dGltZXMuYXBwZW5kKHRpbWUucGVyZl9jb3VudGVyKCktc3RhcnRbMF0pCiAgICAgICAgZGVmIG9uX3NhdmUodHJhaW5lcik6CiAgICAgICAgICAgIHN0YXRlPXNhdmVfY2hlY2twb2ludF9nZW5lcmF0aW9uKHRyYWluZXIscGVyc2lzdGVudCxtZXRhZGF0YSk7c2F2ZWQuYXBwZW5kKHN0YXRlKQogICAgICAgICAgICBmb3IgbmFtZSBpbiAoJ3Jlc3VsdHMuY3N2JywnYXJncy55YW1sJyk6CiAgICAgICAgICAgICAgICBwPVBhdGgodHJhaW5lci5zYXZlX2RpcikvbmFtZQogICAgICAgICAgICAgICAgaWYgcC5leGlzdHMoKTphdG9taWNfY29weShwLHBlcnNpc3RlbnQvJ3RyYWluaW5nJy9uYW1lKQogICAgICAgICAgICBtbGZsb3cubG9nX21ldHJpY3Moe2sucmVwbGFjZSgnKCcsJ18nKS5yZXBsYWNlKCcpJywnXycpOmZsb2F0KHYpIGZvciBrLHYgaW4gdHJhaW5lci5tZXRyaWNzLml0ZW1zKCl9LHN0ZXA9aW50KHRyYWluZXIuZXBvY2gpKQogICAgICAgICAgICBzbmFwc2hvdF90cmFja2luZyhkYXRhYmFzZSxhcnRpZmFjdF9kaXIscGVyc2lzdGVudCkKICAgICAgICBtb2RlbC5hZGRfY2FsbGJhY2soJ29uX3RyYWluX2JhdGNoX3N0YXJ0JyxiYXRjaF9zdGFydCk7bW9kZWwuYWRkX2NhbGxiYWNrKCdvbl90cmFpbl9iYXRjaF9lbmQnLGJhdGNoX2VuZCkKICAgICAgICBtb2RlbC5hZGRfY2FsbGJhY2soJ29uX21vZGVsX3NhdmUnLG9uX3NhdmUpCiAgICAgICAgc3RhcnRlZD10aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgdHJ5OgogICAgICAgICAgICBpZiByZXN1bWU6cmVzdWx0PW1vZGVsLnRyYWluKHJlc3VtZT1UcnVlLGRldmljZT0nMCcpCiAgICAgICAgICAgIGVsc2U6cmVzdWx0PW1vZGVsLnRyYWluKGRhdGE9c3RyKHJ1bnRpbWUpLHByb2plY3Q9c3RyKHdvcmsvJ3J1bnMnKSxuYW1lPXJ1bl9pZCxleGlzdF9vaz1GYWxzZSxzYXZlX3BlcmlvZD0tMSwqKntrOnYgZm9yIGssdiBpbiBwYXJhbXMuaXRlbXMoKSBpZiBrIT0nbW9kZWwnfSkKICAgICAgICAgICAgbGFzdCxzdGF0ZT12ZXJpZnlfbGF0ZXN0KHBlcnNpc3RlbnQpCiAgICAgICAgICAgICMgVmVyaWZ5IHRoZSByZWNvdmVyeSBwYXlsb2FkIGFjdHVhbGx5IGNvbnRhaW5zIGVwb2NoICsgb3B0aW1pemVyIHN0YXRlLCB3aXRob3V0IGxvYWRpbmcgQ1VEQSB0ZW5zb3JzLgogICAgICAgICAgICBwYXlsb2FkPXRvcmNoLmxvYWQobGFzdCxtYXBfbG9jYXRpb249J2NwdScsd2VpZ2h0c19vbmx5PUZhbHNlKQogICAgICAgICAgICByZWNvdmVyYWJsZT1wYXlsb2FkLmdldCgnb3B0aW1pemVyJykgaXMgbm90IE5vbmUgYW5kIHBheWxvYWQuZ2V0KCdlcG9jaCcsLTEpPj0wCiAgICAgICAgICAgIGRlbCBwYXlsb2FkCiAgICAgICAgICAgIHJlY29yZD17KiptZXRhZGF0YSwnc3RhdHVzJzonY29tcGxldGVkJywnZ3B1JzpncHUsJ2VsYXBzZWRfc2Vjb25kcyc6dGltZS5wZXJmX2NvdW50ZXIoKS1zdGFydGVkLAogICAgICAgICAgICAgICAgJ3BlYWtfY3VkYV9hbGxvY2F0ZWRfR2lCJzp0b3JjaC5jdWRhLm1heF9tZW1vcnlfYWxsb2NhdGVkKCkvMTAyNCoqMywKICAgICAgICAgICAgICAgICdwZWFrX2N1ZGFfcmVzZXJ2ZWRfR2lCJzp0b3JjaC5jdWRhLm1heF9tZW1vcnlfcmVzZXJ2ZWQoKS8xMDI0KiozLAogICAgICAgICAgICAgICAgJ21lYW5fYmF0Y2hfY29tcHV0ZV9zZWNvbmRzX2FmdGVyMTAnOnN1bSh0aW1lc1sxMDpdKS9tYXgoMSxsZW4odGltZXNbMTA6XSkpLAogICAgICAgICAgICAgICAgJ3BlcnNpc3RlbnRfY2hlY2twb2ludF92ZXJpZmllZCc6VHJ1ZSwnb3B0aW1pemVyX3N0YXRlX3ZlcmlmaWVkJzpyZWNvdmVyYWJsZSwKICAgICAgICAgICAgICAgICdzYXZlZF9nZW5lcmF0aW9ucyc6bGVuKHNhdmVkKSwnZGF0YXNldF9pbWFnZXMnOm1hbmlmZXN0Wyd0cmFpbl9pbWFnZXMnXSwKICAgICAgICAgICAgICAgICdxdWFsaXR5X2NsYWltJzpGYWxzZSBpZiBtb2RlPT0nYmVuY2htYXJrJyBlbHNlICdWYWxpZGF0aW9uIG9ubHk7IHNlcGFyYXRlIGZpbmFsIGV2YWx1YXRpb24gcmVxdWlyZWQnLAogICAgICAgICAgICAgICAgJ25vdGUnOidCYXRjaCB0aW1pbmcgZXhjbHVkZXMgbG9hZGVyIHRpbWU7IHVzZSBlcG9jaCByZXN1bHRzLmNzdiBmb3IgZnVsbC1ydW4gRVRBLiBObyA3MCUgcGVyZm9ybWFuY2UgY2xhaW0uJ30KICAgICAgICAgICAgaWYgbW9kZT09J2JlbmNobWFyaycgYW5kIG5vdCByZWNvdmVyYWJsZTpyYWlzZSBSdW50aW1lRXJyb3IoJ0JlbmNobWFyayByZWNvdmVyeSBwYXlsb2FkIGxhY2tzIG9wdGltaXplciBzdGF0ZScpCiAgICAgICAgICAgIG91dHB1dD13b3JrL2Yne21vZGV9X3JlcG9ydC5qc29uJztvdXRwdXQud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlY29yZCxpbmRlbnQ9MikpCiAgICAgICAgICAgIGF0b21pY19jb3B5KG91dHB1dCxwZXJzaXN0ZW50L291dHB1dC5uYW1lKTttbGZsb3cubG9nX2FydGlmYWN0KHN0cihvdXRwdXQpKQogICAgICAgICAgICBpZiBtb2RlPT0ndHJhaW4nOgogICAgICAgICAgICAgICAgZGVzdGluYXRpb249d29yay8neW9sbzExbS13YXN0ZS1iZXN0LnB0JztzaHV0aWwuY29weTIobW9kZWwudHJhaW5lci5iZXN0LGRlc3RpbmF0aW9uKQogICAgICAgICAgICAgICAgYXRvbWljX2NvcHkoZGVzdGluYXRpb24scGVyc2lzdGVudC9kZXN0aW5hdGlvbi5uYW1lKTttbGZsb3cubG9nX2FydGlmYWN0KHN0cihkZXN0aW5hdGlvbiksYXJ0aWZhY3RfcGF0aD0nd2VpZ2h0cycpCiAgICAgICAgICAgIGZvciBwIGluIFBhdGgobW9kZWwudHJhaW5lci5zYXZlX2RpcikuaXRlcmRpcigpOgogICAgICAgICAgICAgICAgaWYgcC5zdWZmaXggaW4gKCcuY3N2JywnLnlhbWwnLCcucG5nJywnLmpwZycpOgogICAgICAgICAgICAgICAgICAgIG1sZmxvdy5sb2dfYXJ0aWZhY3Qoc3RyKHApLGFydGlmYWN0X3BhdGg9J3RyYWluaW5nJykKICAgICAgICAgICAgICAgICAgICBhdG9taWNfY29weShwLHBlcnNpc3RlbnQvJ3RyYWluaW5nJy9wLm5hbWUpCiAgICAgICAgICAgIHByaW50KGpzb24uZHVtcHMocmVjb3JkLGluZGVudD0yKSkKICAgICAgICBmaW5hbGx5OgogICAgICAgICAgICAjIFNRTGl0ZSBzbmFwc2hvdHMgdXNlIGl0cyBiYWNrdXAgQVBJIHJhdGhlciB0aGFuIGNvcHlpbmcgYSBsaXZlIFdBTCBkYXRhYmFzZS4KICAgICAgICAgICAgc25hcHNob3RfdHJhY2tpbmcoZGF0YWJhc2UsYXJ0aWZhY3RfZGlyLHBlcnNpc3RlbnQpCiAgICBzbmFwc2hvdF90cmFja2luZyhkYXRhYmFzZSxhcnRpZmFjdF9kaXIscGVyc2lzdGVudCkKCgppZiBfX25hbWVfXz09J19fbWFpbl9fJzoKICAgIHBhcnNlcj1hcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpO3BhcnNlci5hZGRfYXJndW1lbnQoJ21vZGUnLGNob2ljZXM9WydiZW5jaG1hcmsnLCd0cmFpbiddKTtwYXJzZXIuYWRkX2FyZ3VtZW50KCctLXJvb3QnLHJlcXVpcmVkPVRydWUpO3BhcnNlci5hZGRfYXJndW1lbnQoJy0tcGVyc2lzdGVudCcscmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tYXBwcm92ZS1mdWxsJyxhY3Rpb249J3N0b3JlX3RydWUnKTtwYXJzZXIuYWRkX2FyZ3VtZW50KCctLXJlc3VtZScsYWN0aW9uPSdzdG9yZV90cnVlJykKICAgIGFyZ3M9cGFyc2VyLnBhcnNlX2FyZ3MoKTtydW4oYXJncy5yb290LGFyZ3MucGVyc2lzdGVudCxhcmdzLm1vZGUsYXJncy5hcHByb3ZlX2Z1bGwsYXJncy5yZXN1bWUpCg=='))
sys.path.insert(0, '/content')
import mlops_cloud_runner as runner


## Persistent storage

Sign in and approve Google’s Drive connection yourself if prompted. This uses `MyDrive/MLOPS_EL_cloud` for project checkpoints, reports and MLflow records. If Drive cannot mount or writing fails, the benchmark stops instead of pretending recovery is available.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PERSISTENT='/content/drive/MyDrive/MLOPS_EL_cloud'


## Upload the small benchmark archive

Choose **`mlops-benchmark-labels-v3.tar`** (about 285 MB). Do not upload the full training archive through this widget. Its separate workflow uses a Drive archive copied to runtime disk.

In [ ]:
from google.colab import files
import gc, time
BENCHMARK_ARCHIVE='/content/mlops-benchmark-labels-v3.tar'
# Alternatively upload through Colab's Files sidebar, then run this cell.
if not pathlib.Path(BENCHMARK_ARCHIVE).exists():
    uploaded=files.upload()
    assert 'mlops-benchmark-labels-v3.tar' in uploaded, 'Select the benchmark archive.'
    del uploaded
    gc.collect()
deadline=time.monotonic()+1800
while pathlib.Path(BENCHMARK_ARCHIVE).stat().st_size < 284508160:
    if time.monotonic()>deadline: raise TimeoutError('Upload incomplete; finish uploading and rerun this cell.')
    print('Upload bytes:', pathlib.Path(BENCHMARK_ARCHIVE).stat().st_size, '/ 284508160', flush=True)
    time.sleep(30)
EXPECTED_BENCHMARK_SHA256='388732d00a3fbd81db132da52cc76c5734430c89e8fc05c7e24e796046b80910'
BENCHMARK_ROOT='/content/mlops_benchmark_bundle'
manifest=runner.unpack(BENCHMARK_ARCHIVE, BENCHMARK_ROOT, EXPECTED_BENCHMARK_SHA256)
print('Verified bundle:', manifest['scope'], manifest['train_images'], manifest['val_images'])


## Run only the short benchmark

This measures CUDA memory and speed and checks that a persistent checkpoint includes optimizer/epoch state. Smoke-validation scores are not model-quality evidence. A CUDA out-of-memory error is a failed feasibility result; do not silently alter the full training plan.

In [ ]:
runner.run(BENCHMARK_ROOT, PERSISTENT, mode='benchmark')


In [ ]:
import json, zipfile
benchmark_dir=pathlib.Path(PERSISTENT)/'benchmark'
report=json.loads((benchmark_dir/'benchmark_report.json').read_text())
project_bytes=sum(p.stat().st_size for p in pathlib.Path(PERSISTENT).rglob('*') if p.is_file())
summary={'status':report['status'],'GPU':report['gpu']['name'],'elapsed_seconds':round(report['elapsed_seconds'],2),'reserved_GPU_GiB':round(report['peak_cuda_reserved_GiB'],2),'checkpoint_verified':report['persistent_checkpoint_verified'],'optimizer_state_verified':report['optimizer_state_verified'],'Drive_project_MB':round(project_bytes/1e6,2),'full_training':'DISABLED'}
print(json.dumps(summary,indent=2))
print('Report:',benchmark_dir/'benchmark_report.json')
evidence_zip=pathlib.Path('/content/mlops-benchmark-evidence.zip')
with zipfile.ZipFile(evidence_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in benchmark_dir.rglob('*'):
        if p.is_file() and p.suffix not in ('.pt','.part'):
            z.write(p,p.relative_to(benchmark_dir))
print('Evidence archive:',evidence_zip)


In [ ]:
"""Engineering-only stop/resume drill, isolated from full training and benchmark records."""
import json
from pathlib import Path
import subprocess
import sys


def main():
    project = Path('/content/drive/MyDrive/MLOPS_EL_cloud')
    persistent = project / 'recovery-drill'
    work = Path('/content/mlops_work/benchmark')
    preserved = work.with_name('benchmark-before-recovery-drill')
    first_session = work.with_name('recovery-drill-session1')
    second_session = work.with_name('recovery-drill-session2')
    for path in (persistent, preserved, first_session, second_session):
        if path.exists():
            raise FileExistsError(f'Drill already exists; review it rather than overwriting: {path}')
    if work.exists():
        work.rename(preserved)
    common = """
import sys
sys.path.insert(0, '/content')
import mlops_cloud_runner as runner
ROOT='/content/mlops_benchmark_bundle'
PERSISTENT='/content/drive/MyDrive/MLOPS_EL_cloud/recovery-drill'
"""
    first = common + """
original_save=runner.save_checkpoint_generation
def stop_after_saved_epoch(trainer, persistent, metadata):
    state=original_save(trainer, persistent, metadata)
    trainer.stop=True
    return state
runner.save_checkpoint_generation=stop_after_saved_epoch
runner.run(ROOT, PERSISTENT, mode='benchmark')
"""
    second = common + "runner.run(ROOT, PERSISTENT, mode='benchmark', resume=True)\n"
    try:
        # Separate interpreters close all SQLite connections and remove in-memory model state.
        subprocess.run([sys.executable, '-c', first], check=True)
        state_path = persistent / 'benchmark/checkpoints/latest.json'
        before = json.loads(state_path.read_text())
        assert before['epoch'] == 1, 'The controlled stop must preserve epoch 1'
        work.rename(first_session)
        subprocess.run([sys.executable, '-c', second], check=True)
        after = json.loads(state_path.read_text())
        report = json.loads((persistent / 'benchmark/benchmark_report.json').read_text())
        assert after['epoch'] == 2, 'Training must resume and finish epoch 2'
        assert before['run_id'] == after['run_id'], 'MLflow run continuity must be preserved'
        assert before['bundle_fingerprint'] == after['bundle_fingerprint']
        assert report['persistent_checkpoint_verified'] and report['optimizer_state_verified']
        record = {'status': 'passed', 'method': 'controlled epoch stop, fresh Python process and empty local work directory',
                  'before_epoch': before['epoch'], 'after_epoch': after['epoch'],
                  'run_id': after['run_id'], 'bundle_fingerprint': after['bundle_fingerprint'],
                  'same_mlflow_run': True, 'optimizer_state_verified': True,
                  'not_a_provider_disconnect_test': True, 'quality_claim': False}
        (persistent / 'recovery_drill_report.json').write_text(json.dumps(record, indent=2))
        print('RECOVERY_DRILL_RESULT', json.dumps(record), flush=True)
    finally:
        if work.exists():
            work.rename(second_session)
        if preserved.exists():
            preserved.rename(work)


if __name__ == '__main__':
    main()


## Full training — authorized 6 November 2026

The user approved YOLO11m, 960 pixels, batch 4, up to 100 epochs with patience 20, on a free GPU. All 7000 training and 1500 validation images are used; test data is excluded. The controller waits for the complete versioned archive in Drive and verifies it before training.

Each epoch saves two generations of verified recovery checkpoints and MLflow snapshots. For another session, reconnect a free GPU, run setup cells 1, 2, 3 and 5, then this full-run cell. Skip completed benchmark and recovery-drill cells. The controller detects and resumes an existing checkpoint automatically.

Training was authorized; the 70% validation mAP@50 target is not guaranteed. Existing baseline results remain preserved.


In [ ]:
"""Start/resume the user-authorized full experiment after persistent input verification."""
import json
import gc
import importlib.metadata
from pathlib import Path
import shutil
import subprocess
import sys
import time
import uuid

import mlops_cloud_runner as runner

PROJECT = Path('/content/drive/MyDrive/MLOPS_EL_cloud')
ARCHIVE = PROJECT / 'mlops-trainval-labels-v3.tar'
ARCHIVE_BYTES = 6700103680
ARCHIVE_SHA256 = '37c7e50b954a996519fc8d4f52715fc93c733b313374ab4f2b91af9725064098'
ROOT = Path('/content/mlops_trainval_bundle')


def main():
    drill_path = PROJECT / 'recovery-drill/recovery_drill_report.json'
    assert drill_path.is_file(), 'Complete checkpoint recovery verification first.'
    drill = json.loads(drill_path.read_text())
    assert drill['status'] == 'passed' and drill['same_mlflow_run']
    completed = PROJECT / 'train/train_report.json'
    if completed.exists() and json.loads(completed.read_text()).get('status') == 'completed':
        print('This full experiment is already complete. Review it before starting another run.')
        return
    deadline = time.monotonic() + 7200
    while not ARCHIVE.is_file() or ARCHIVE.stat().st_size != ARCHIVE_BYTES:
        if time.monotonic() > deadline:
            raise TimeoutError('Full archive upload is incomplete; finish the Drive upload and rerun.')
        print('Waiting for complete persistent dataset upload:', ARCHIVE, flush=True)
        time.sleep(30)
    if not ROOT.exists():
        local_archive = Path('/content/mlops-trainval-labels-v3.tar')
        assert shutil.disk_usage('/content').free > 2 * ARCHIVE_BYTES + 5 * 1024**3, 'Insufficient runtime disk headroom'
        print('Copying archive from Drive to local runtime disk.', flush=True)
        runner.atomic_copy(ARCHIVE, local_archive)
        print('Checking archive and every payload file before training.', flush=True)
        runner.unpack(local_archive, ROOT, ARCHIVE_SHA256)
    manifest = runner.verify_bundle(ROOT)
    assert manifest['scope'] == 'trainval'
    assert manifest['train_images'] == 7000 and manifest['val_images'] == 1500
    # Refuse to overwrite incomplete evidence from a start that never produced a recovery checkpoint.
    checkpoint = PROJECT / 'train/checkpoints/latest.json'
    mode_dir = PROJECT / 'train'
    resume = checkpoint.is_file()
    if mode_dir.exists() and not resume:
        raise RuntimeError('An earlier attempt exists without a checkpoint. Inspect it before retrying.')
    # Preserve each session's environment independently of the runner's latest MLflow snapshot.
    import torch
    versions = {name: importlib.metadata.version(name) for name in ('torch', 'torchvision', 'ultralytics', 'mlflow')}
    sessions = PROJECT / 'train-session-records'
    previous = sorted(sessions.glob('*/session.json')) if sessions.exists() else []
    if resume and previous:
        initial = json.loads(previous[0].read_text())['versions']
        assert versions == initial, f'Cloud dependency versions changed: {versions}; restore {initial} before resuming.'
    session = sessions / (time.strftime('%Y%m%d-%H%M%S', time.gmtime()) + '-' + uuid.uuid4().hex[:8])
    session.mkdir(parents=True)
    record = {'versions': versions, 'gpu': torch.cuda.get_device_name(0), 'resume': resume,
              'archive_sha256': ARCHIVE_SHA256, 'runner_sha256': runner.sha(Path(runner.__file__)),
              'bundle_fingerprint': runner.sha(ROOT / 'manifest.json')}
    (session / 'session.json').write_text(json.dumps(record, indent=2))
    (session / 'requirements-resolved.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
    runner.atomic_copy(ROOT / 'config.yaml', session / 'config.yaml')
    runner.atomic_copy(ROOT / 'manifest.json', session / 'manifest.json')
    if resume:
        runner.atomic_copy(checkpoint, session / 'resume-from.json')
    approval = {'user_authorized_on': '2026-11-06', 'model': 'YOLO11m',
                'epochs_max': 100, 'patience': 20, 'imgsz': 960, 'batch': 4,
                'free_gpu_only': True, 'archive_sha256': ARCHIVE_SHA256,
                'runner_sha256': runner.sha(Path(runner.__file__)),
                'dataset_approval': 'faculty approval pending',
                'target_validation_map50': 0.70, 'target_is_not_a_guarantee': True}
    (PROJECT / 'full_training_authorization.json').write_text(json.dumps(approval, indent=2))
    print('Starting full experiment:', 'RESUME' if resume else 'NEW', '7000 train / 1500 validation; test excluded.', flush=True)
    gc.collect()
    torch.cuda.empty_cache()
    runner.run(ROOT, PROJECT, mode='train', approve_full=True, resume=resume)


if __name__ == '__main__':
    main()
